In [1]:
import numpy as np


In [2]:
import sys
sys.path.append('../')
from pqcqec.noise.builder import build_regular_noisy_circuit, create_pqc_circuit_template_simplified, update_pqc_circuit_template, decompile_circuit
from pqcqec.circuits.generate import generate_random_circuit


In [3]:
NUM_QUBITS = 2
NUM_GATES = 4
NUM_GATE_BLOCKS = 2

# User controls PQC blocks exactly - no automatic final block
# PQC inserted after every NUM_GATE_BLOCKS gates
PQC_BLOCKS = NUM_GATES // NUM_GATE_BLOCKS  # 4 // 2 = 2

## Configuration

**PQC Placement Strategy:**
- `PQC_BLOCKS = NUM_GATES // NUM_GATE_BLOCKS` 
- PQC inserted **only** after every `NUM_GATE_BLOCKS` logical gates
- **No automatic final block** - you have full control!
  
For 4 gates with `gate_blocks=2`:
- PQC after gate 1 (index 1) ✓
- PQC after gate 3 (index 3) ✓
- Total: **2 PQC blocks**

In [4]:
circuit = generate_random_circuit(NUM_QUBITS, NUM_GATES, seed=42, backend='list')
print("Generated Circuit:")
print(circuit)

Generated Circuit:
[('cx', [0, 1], []), ('x', [1], []), ('z', [0], []), ('z', [0], [])]


In [5]:
x_noise = np.random.normal(0, 0.01, size=(NUM_GATES,))
z_noise = np.random.normal(0, 0.01, size=(NUM_GATES,))

noisy_circ = build_regular_noisy_circuit(circuit, x_noise=x_noise, z_noise=z_noise, return_tagged=True)
print("Noisy Circuit with Tagged Noise:")
for op in noisy_circ:
    print(op)

Noisy Circuit with Tagged Noise:
('cx', [0, 1], [])
('rx', [0], [np.float64(-0.009627408358968496)], {'noise': True})
('rz', [0], [np.float64(0.003129000945771592)], {'noise': True})
('rx', [1], [np.float64(-0.009627408358968496)], {'noise': True})
('rz', [1], [np.float64(0.003129000945771592)], {'noise': True})
('x', [1], [])
('rx', [1], [np.float64(0.012582651636851058)], {'noise': True})
('rz', [1], [np.float64(0.01116161394117716)], {'noise': True})
('z', [0], [])
('rx', [0], [np.float64(-0.007223209396238216)], {'noise': True})
('rz', [0], [np.float64(0.008663511753666909)], {'noise': True})
('z', [0], [])
('rx', [0], [np.float64(-0.0038590390929312186)], {'noise': True})
('rz', [0], [np.float64(-0.0034613325693882663)], {'noise': True})


## Build PQC Circuit Template

**Two functions with different purposes:**

1. **`build_circuit_with_pqc()`** - Direct builder (one-time use)
   - Takes `pqc_params` array
   - Returns compiled circuit immediately
   - Use when you only build once

2. **`create_pqc_circuit_template()`** - Template builder (reusable)
   - Takes `num_pqc_blocks` (number)
   - Returns template **dictionary** with structure info
   - Use with `update_pqc_circuit_template()` for fast updates
   - **Perfect for training loops!**

We're using the **template approach** here for performance.

In [6]:
from pqcqec.noise.builder import decompile_circuit

# Use create_pqc_circuit_template instead of build_circuit_with_pqc
pqc_noisy_circ_template = create_pqc_circuit_template_simplified(
    noisy_circ, num_qubits=NUM_QUBITS, gate_blocks=NUM_GATE_BLOCKS, 
    pqc_gates=['rz', 'rx', 'rz'], num_pqc_blocks=PQC_BLOCKS, dtype=np.float32, ignore_noise_gates=True)

print("PQC Circuit Template:")
print(f"  Total gates: {len(pqc_noisy_circ_template['gate_ids'])}")
print(f"  PQC param map shape: {pqc_noisy_circ_template['pqc_param_map'].shape}")

# Decompile to see gate order
template_ops = decompile_circuit(
    pqc_noisy_circ_template['gate_ids'],
    pqc_noisy_circ_template['wire1'],
    pqc_noisy_circ_template['wire2'],
    pqc_noisy_circ_template['theta']
)

print(f"\nGate order (first 20 gates):")
for i, op in enumerate(template_ops[:20]):
    print(f"  {i}: {op[0]:4s} {op[1]}")


PQC Circuit Template:
  Total gates: 26
  PQC param map shape: (12, 4)

Gate order (first 20 gates):
  0: cx   [0, 1]
  1: rx   [0]
  2: rz   [0]
  3: rx   [1]
  4: rz   [1]
  5: x    [1]
  6: rx   [1]
  7: rz   [1]
  8: rz   [0]
  9: rx   [0]
  10: rz   [0]
  11: rz   [1]
  12: rx   [1]
  13: rz   [1]
  14: z    [0]
  15: rx   [0]
  16: rz   [0]
  17: z    [0]
  18: rx   [0]
  19: rz   [0]


## Test Template Update with Random PQC Parameters

In [7]:
# Generate random PQC parameters
pqc_params = np.random.randn(PQC_BLOCKS, NUM_QUBITS, 3).astype(np.float32)
print(f"PQC Parameters shape: {pqc_params.shape}")
print(f"Expected: ({PQC_BLOCKS}, {NUM_QUBITS}, 3)")

# Update template with new parameters
gate_ids, wire1, wire2, theta = update_pqc_circuit_template(pqc_noisy_circ_template, pqc_params)

print(f"\nUpdated Circuit:")
print(f"  Total gates: {len(gate_ids)}")
print(f"  Logical gates: {NUM_GATES}")
print(f"  Noise gates: {len(noisy_circ) - NUM_GATES}")
print(f"  PQC gates: {PQC_BLOCKS * NUM_QUBITS * 3}")
print(f"  Expected total: {len(noisy_circ) + PQC_BLOCKS * NUM_QUBITS * 3}")

PQC Parameters shape: (2, 2, 3)
Expected: (2, 2, 3)

Updated Circuit:
  Total gates: 26
  Logical gates: 4
  Noise gates: 10
  PQC gates: 12
  Expected total: 26


## Performance Comparison: Template vs Full Rebuild

In [8]:
import time
from pqcqec.noise.builder import build_circuit_with_pqc

num_iterations = 10000

# Method 1: Template updates
params_list = [np.random.randn(PQC_BLOCKS, NUM_QUBITS, 3).astype(np.float32) for _ in range(num_iterations)]

start = time.perf_counter()
for params in params_list:
    g, w1, w2, theta = update_pqc_circuit_template(pqc_noisy_circ_template, params)
end = time.perf_counter()
template_time = (end - start) / num_iterations

print(f"Template Update Method:")
print(f"  Average time: {template_time*1000:.4f} ms")
print(f"  Throughput: {1/template_time:.0f} updates/sec")

# Method 2: Full rebuild
start = time.perf_counter()
for params in params_list:
    g, w1, w2, theta = build_circuit_with_pqc(
        noisy_circ, NUM_QUBITS, NUM_GATE_BLOCKS, ['rz', 'rx', 'rz'], params,
        return_numba=True, ignore_noise_gates=True
    )
end = time.perf_counter()
rebuild_time = (end - start) / num_iterations

print(f"\nFull Rebuild Method:")
print(f"  Average time: {rebuild_time*1000:.4f} ms")
print(f"  Throughput: {1/rebuild_time:.0f} updates/sec")

speedup = rebuild_time / template_time
print(f"\n{'='*60}")
print(f"SPEEDUP: {speedup:.1f}x faster with template!")
print(f"{'='*60}")

Template Update Method:
  Average time: 0.0029 ms
  Throughput: 340943 updates/sec

Full Rebuild Method:
  Average time: 0.0357 ms
  Throughput: 27975 updates/sec

SPEEDUP: 12.2x faster with template!

Full Rebuild Method:
  Average time: 0.0357 ms
  Throughput: 27975 updates/sec

SPEEDUP: 12.2x faster with template!
